# NBA Similarity Data Analytics Project
## Sahil Sardar
### Using the Basketball-Reference downloaded csv of data from 2023-2027 to find out which players are the most similar to each other
#### USAGE:
##### Front offices looking into making changes/replacements to fill positions analytically

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import sklearn as skl

## Loading the data
One row per player per season, using  `load_bbref.py` from the Basketball-Reference downloads.

In [3]:
player_season = pd.read_csv("data/processed/player_seasons.csv")
print(player_season.shape)
player_season.head()

(1644, 65)


,SEASON,PLAYER_ID,PLAYER_NAME,TEAM_ABBREVIATION,AGE,POS,GP,MIN_TOTAL,FGM_P100,FGA_P100,...,SHOT_FG_10_16,SHOT_FG_16_3P,SHOT_FG_3P,PCT_FG_ASSISTED_2P,PCT_FG_ASSISTED_3P,DUNK_FGA_PCT,CORNER3_SHARE_OF_3PA,MPG,IS_PARTIAL_SEASON,MIN_OK
0,2023-24,DeMar DeRozan,DeMar DeRozan,CHI,34.0,SF,79.0,2989.0,10.8,22.6,...,0.490,0.402,0.333,0.282,0.853,0.019,0.316,37.835443,False,True
1,2023-24,Domantas Sabonis,Domantas Sabonis,SAC,27.0,C,82.0,2928.0,10.5,17.7,...,0.467,0.471,0.379,0.607,0.909,0.113,0.057,35.707317,False,True
2,2023-24,Coby White,Coby White,CHI,23.0,PG,79.0,2881.0,9.4,20.9,...,0.485,0.267,0.376,0.289,0.833,0.012,0.126,36.468354,False,True
3,2023-24,Mikal Bridges,Mikal Bridges,BRK,27.0,SF,82.0,2854.0,9.8,22.5,...,0.468,0.292,0.372,0.494,0.882,0.016,0.338,34.804878,False,True
4,2023-24,Paolo Banchero,Paolo Banchero,ORL,21.0,PF,80.0,2799.0,11.3,24.9,...,0.395,0.421,0.339,0.363,0.655,0.063,0.068,34.987500,False,True


# Data Dictionary

| Column Name             | Data Type | Description |
|:------------------------|:----------|:------------|
| SEASON                  |           |             |
| PLAYER_ID               |           |             |
| PLAYER_NAME             |           |             |
| TEAM_ABBREVIATION       |           |             |
| AGE                     |           |             |
| POS                     |           |             |
| GP                      |           |             |
| MIN_TOTAL               |           |             |
| FGM_P100                |           |             |
| FGA_P100                |           |             |
| FG3M_P100               |           |             |
| FG3A_P100               |           |             |
| FTM_P100                |           |             |
| FTA_P100                |           |             |
| OREB_P100               |           |             |
| DREB_P100               |           |             |
| REB_P100                |           |             |
| AST_P100                |           |             |
| STL_P100                |           |             |
| BLK_P100                |           |             |
| TOV_P100                |           |             |
| PF_P100                 |           |             |
| PTS_P100                |           |             |
| FG3_PCT                 |           |             |
| FT_PCT                  |           |             |
| OFF_RATING              |           |             |
| DEF_RATING              |           |             |
| PER                     |           |             |
| TS_PCT                  |           |             |
| FG3A_RATE               |           |             |
| FT_RATE                 |           |             |
| OREB_PCT                |           |             |
| DREB_PCT                |           |             |
| REB_PCT                 |           |             |
| AST_PCT                 |           |             |
| STL_PCT                 |           |             |
| BLK_PCT                 |           |             |
| TM_TOV_PCT              |           |             |
| USG_PCT                 |           |             |
| OWS                     |           |             |
| DWS                     |           |             |
| WS                      |           |             |
| WS_48                   |           |             |
| OBPM                    |           |             |
| DBPM                    |           |             |
| BPM                     |           |             |
| VORP                    |           |             |
| AVG_SHOT_DIST           |           |             |
| SHOT_FREQ_0_3           |           |             |
| SHOT_FREQ_3_10          |           |             |
| SHOT_FREQ_10_16         |           |             |
| SHOT_FREQ_16_3P         |           |             |
| SHOT_FREQ_3P            |           |             |
| SHOT_FG_0_3             |           |             |
| SHOT_FG_3_10            |           |             |
| SHOT_FG_10_16           |           |             |
| SHOT_FG_16_3P           |           |             |
| SHOT_FG_3P              |           |             |
| PCT_FG_ASSISTED_2P      |           |             |
| PCT_FG_ASSISTED_3P      |           |             |
| DUNK_FGA_PCT            |           |             |
| CORNER3_SHARE_OF_3PA    |           |             |
| MPG                     |           |             |
| IS_PARTIAL_SEA          |           |             |

## Filter to reliable samples
Players with very few minutes have noisy stats (a 30-minute player can look like a perfect match for a star). `MIN_OK` is true for 500+ minutes in a full season, or 150+ in the partial 2026-27 season.

In [4]:
before = player_season.groupby("SEASON").size().rename("all_players")

player_season_filtered = player_season[player_season["MIN_OK"]].copy()

after = player_season_filtered.groupby("SEASON").size().rename("kept")
summary = pd.concat([before, after], axis=1)
summary["dropped"] = summary["all_players"] - summary["kept"]
display(summary)
print(f"{len(player_season_filtered)} of {len(player_season)} player-seasons kept ({len(player_season_filtered)/len(player_season):.0%})") 
# How many seasons count for each player? 
player_season_filtered.groupby("PLAYER_ID")["SEASON"].nunique().value_counts().sort_index().rename_axis("seasons_played").rename("players")

,all_players,kept,dropped
SEASON,,,
2023-24,376,360,16
2024-25,392,375,17
2025-26,393,379,14
2026-27,483,312,171


1426 of 1644 player-seasons kept (87%)


seasons_played
1    140
2    119
3    120
4    172
Name: players, dtype: int64